In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Project Configuration

In [ ]:
import os, subprocess, json


BASE = "/content/drive/MyDrive/7006SCN"
DATA = f"{BASE}/data"
PROC = f"{BASE}/processed"
MODELS = f"{BASE}/models"
META = f"{BASE}/metadata"
OUTPUTS = f"{BASE}/outputs"

# ----------------------------------
# Artefact paths
# ----------------------------------
RAW_DATA_PATH = f"{DATA}/taxi.csv"

PROC_TRAIN = f"{PROC}/training.parquet"
PROC_TEST = f"{PROC}/test.parquet"

PIPELINE_PATH = f"{MODELS}/preprocessing_pipeline"

LR_MODEL_PATH = f"{MODELS}/lr_model"
RF_MODEL_PATH = f"{MODELS}/rf_model"
GBT_MODEL_PATH = f"{MODELS}/gbt_model"

TASK1_META = f"{META}/task1_metadata.json"
TASK2_META = f"{META}/task2_metadata.json"
TASK3_META = f"{META}/task3_metadata.json"


for p in [DATA, PROC, MODELS, META, OUTPUTS]:
    os.makedirs(p, exist_ok=True)

def verify_exists(path, label=""):
    """subprocess verify — prints ls -lh for the path"""

    result = subprocess.run(
        ["ls", "-lh", path],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print(f"✓ {label or path}:")
        print(result.stdout.strip())
    else:
        raise FileNotFoundError(
            f"NOT FOUND: {path}\n{result.stderr}"
        )

print("Shared constants loaded ✓")

Shared constants loaded ✓


##Installing PySpark

In [ ]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("7006SCN_Task3").master("local[*]" )
    .config("spark.driver.memory","8g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Session started:", spark.sparkContext.applicationId)

spark

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,221 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,306 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,024 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InReleas

# Loading Task2 pipeline + splits

In [ ]:
from pyspark.ml import PipelineModel

# -------------------------------------------
# Verifying artefacts exist (subprocess only)
# -------------------------------------------

verify_exists(PIPELINE_PATH, "Preprocessing PipelineModel")
verify_exists(PROC_TRAIN, "Training Parquet split")
verify_exists(PROC_TEST, "Test Parquet split")
verify_exists(TASK2_META, "Task2 metadata JSON")


# ----------------------------------
# Loading Task2 metadata
# ----------------------------------

with open(TASK2_META) as f:
  t2 = json.load(f)
  LABEL_COL = t2["target_col"]
  print(f"Target column : {LABEL_COL}")
  print(f"Train rows : {t2['train_rows']:,}")
  print(f"Pipeline stages: {t2['pipeline_stages']}")

print(t2)

# ----------------------------------
# Loading PipelineModel — PySpark
# ----------------------------------

preproc_model = PipelineModel.load(PIPELINE_PATH)
print(f"Pipeline loaded: {len(preproc_model.stages)} stages")


# ----------------------------------
# Loading Parquet splits
# ----------------------------------

train_df = spark.read.parquet(PROC_TRAIN).cache()
test_df = spark.read.parquet(PROC_TEST).cache()
print(f"Train: {train_df.count():,} Test: {test_df.count():,}")

train_fe = preproc_model.transform(train_df)
test_fe = preproc_model.transform(test_df)

train_fe.cache()
test_fe.cache()

print(train_fe.count(), test_fe.count())

✓ Preprocessing PipelineModel:
total 8.0K
drwx------  2 root root 4.0K Jun 14 14:48 metadata
drwx------ 12 root root 4.0K Jun 14 14:48 stages
✓ Training Parquet split:
total 127M
-rw------- 1 root root 644K Jun 14 14:48 part-00000-6df02b93-6ab8-484b-992e-3032afc29a4e-c000.snappy.parquet
-rw------- 1 root root 648K Jun 14 14:48 part-00001-6df02b93-6ab8-484b-992e-3032afc29a4e-c000.snappy.parquet
-rw------- 1 root root 651K Jun 14 14:48 part-00002-6df02b93-6ab8-484b-992e-3032afc29a4e-c000.snappy.parquet
-rw------- 1 root root 643K Jun 14 14:48 part-00003-6df02b93-6ab8-484b-992e-3032afc29a4e-c000.snappy.parquet
-rw------- 1 root root 646K Jun 14 14:48 part-00004-6df02b93-6ab8-484b-992e-3032afc29a4e-c000.snappy.parquet
-rw------- 1 root root 650K Jun 14 14:48 part-00005-6df02b93-6ab8-484b-992e-3032afc29a4e-c000.snappy.parquet
-rw------- 1 root root 651K Jun 14 14:48 part-00006-6df02b93-6ab8-484b-992e-3032afc29a4e-c000.snappy.parquet
-rw------- 1 root root 650K Jun 14 14:48 part-00007-6df02b

# Creating evaluators

In [ ]:
# -----------------------------------------
# Evaluator shared by all regression models
# -----------------------------------------

from pyspark.ml.evaluation import RegressionEvaluator

reg_evaluator = RegressionEvaluator(
    labelCol="Trip_Seconds",
    predictionCol="prediction"
)

In [ ]:
rmse_eval = RegressionEvaluator(
    labelCol="Trip_Seconds",
    predictionCol="prediction",
    metricName="rmse"
)

mae_eval = RegressionEvaluator(
    labelCol="Trip_Seconds",
    predictionCol="prediction",
    metricName="mae"
)

r2_eval = RegressionEvaluator(
    labelCol="Trip_Seconds",
    predictionCol="prediction",
    metricName="r2"
)

In [ ]:
# ----------------------------------
# Sample for hyperparameter tuning
# ----------------------------------

from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import time

train_sample = train_fe.sample(0.02, seed=42).cache()
train_sample.count()

158679

# Model 1 - Linear Regression

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline

lr = LinearRegression(
    featuresCol="features",
    labelCol="Trip_Seconds",
    maxIter=20
)

paramGrid_lr = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 1.0])
    .build()
)

lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid_lr,
    evaluator=rmse_eval,
    numFolds=2,
    parallelism=2
)

t0 = time.time()
lr_model = lr_cv.fit(train_sample)
time_lr = time.time() - t0

preds_lr = lr_model.transform(test_fe)

rmse_lr = rmse_eval.evaluate(preds_lr)
r2_lr   = r2_eval.evaluate(preds_lr)
mae_lr  = mae_eval.evaluate(preds_lr)

best_lr = lr_model.bestModel
print("Best params:", {str(k): v for k, v in best_lr.extractParamMap().items() if 'regParam' in str(k) or 'elasticNet' in str(k)})

print(f"LR | RMSE={rmse_lr:.2f} | R2={r2_lr:.4f} | MAE={mae_lr:.2f} | time={time_lr:.2f}s")

Best params: {'LinearRegression_abff4e185115__elasticNetParam': 1.0, 'LinearRegression_abff4e185115__regParam': 0.1}
LR | RMSE=1487.83 | R2=0.1943 | MAE=456.32 | time=109.47s


In [ ]:
# ----------------------------------
# Coefficients
# ----------------------------------

coef_list = best_lr.coefficients.toArray().tolist()

coef_df = spark.createDataFrame(
    [(i, float(c)) for i, c in enumerate(coef_list)],
    ["feature_index", "coefficient"]
)

coef_df.show(10)

+-------------+-------------------+
|feature_index|        coefficient|
+-------------+-------------------+
|            0|  494.4132060871786|
|            1| 11.610411515762758|
|            2| -80.64508871341906|
|            3|  5.934421614938517|
|            4|-23.742680702390448|
|            5| 59.885494476977634|
|            6|  40.34824191659529|
|            7| 28.048771416571455|
|            8|   95.7106255782038|
|            9|-26.915728589110994|
+-------------+-------------------+
only showing top 10 rows


# Model 2 - Decision Tree regressor

In [ ]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="Trip_Seconds"
)

paramGrid_dt = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [5, 10])
    .build()
)

dt_cv = CrossValidator(
    estimator=dt,
    estimatorParamMaps=paramGrid_dt,
    evaluator=rmse_eval,
    numFolds=2,
    parallelism=2
)

t0 = time.time()
dt_model = dt_cv.fit(train_sample)
time_dt = time.time() - t0

preds_dt = dt_model.transform(test_fe)

rmse_dt = rmse_eval.evaluate(preds_dt)
r2_dt   = r2_eval.evaluate(preds_dt)
mae_dt  = mae_eval.evaluate(preds_dt)

best_dt = dt_model.bestModel
print("Best params:", {str(k): v for k, v in best_dt.extractParamMap().items() if 'numTrees' in str(k) or 'maxDepth' in str(k)})

print(f"DT | RMSE={rmse_dt:.2f} | R2={r2_dt:.4f} | MAE={mae_dt:.2f} | time={time_dt:.2f}s")

Best params: {'DecisionTreeRegressor_808522cb37d2__maxDepth': 5}
DT | RMSE=1479.46 | R2=0.2034 | MAE=425.61 | time=62.99s


# Model 3 - Random Forest Regressor

In [ ]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="Trip_Seconds",
    numTrees=10
)

paramGrid_rf = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [10, 20])
    .addGrid(rf.maxDepth, [5])
    .build()
)

rf_cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=paramGrid_rf,
    evaluator=r2_eval,
    numFolds=3,
    parallelism=2
)

t0 = time.time()
rf_model = rf_cv.fit(train_sample)
time_rf = time.time() - t0

preds_rf = rf_model.transform(test_fe)

rmse_rf = rmse_eval.evaluate(preds_rf)
r2_rf   = r2_eval.evaluate(preds_rf)
mae_rf  = mae_eval.evaluate(preds_rf)

best_rf = rf_model.bestModel
print("Best params:", {str(k): v for k, v in best_rf.extractParamMap().items() if 'numTrees' in str(k) or 'maxDepth' in str(k)})

print(f"RF | RMSE={rmse_rf:.2f} | R2={r2_rf:.4f} | MAE={mae_rf:.2f} | time={time_rf:.2f}s")

Best params: {'RandomForestRegressor_4d2b2b4c00db__maxDepth': 5, 'RandomForestRegressor_4d2b2b4c00db__numTrees': 20}
RF | RMSE=1464.82 | R2=0.2191 | MAE=418.12 | time=128.72s


In [ ]:
# ----------------------------------
# Feature importance
# ----------------------------------

fi = best_rf.featureImportances.toArray()

fi_df = spark.createDataFrame(
    [(i, float(v)) for i, v in enumerate(fi)],
    ["feature_index", "importance"]
)

fi_df.show(10)


+-------------+--------------------+
|feature_index|          importance|
+-------------+--------------------+
|            0|  0.4190790309978974|
|            1| 0.03946386865347319|
|            2| 0.14719345624175434|
|            3|0.017853372286416307|
|            4| 0.08572691692013318|
|            5|0.042567500126366016|
|            6|0.005111538030890352|
|            7|0.002939194902941...|
|            8|0.011835491276715786|
|            9|4.316189710278782E-4|
+-------------+--------------------+
only showing top 10 rows


# Model 4 - GBT Regressor

In [ ]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="Trip_Seconds",
    maxIter=20
)

paramGrid_gbt = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5])
    .build()
)

gbt_cv = CrossValidator(
    estimator=gbt,
    estimatorParamMaps=paramGrid_gbt,
    evaluator=rmse_eval,
    numFolds=2,
    parallelism=2
)

t0 = time.time()
gbt_model = gbt_cv.fit(train_sample)
time_gbt = time.time() - t0

preds_gbt = gbt_model.transform(test_fe)

rmse_gbt = rmse_eval.evaluate(preds_gbt)
r2_gbt   = r2_eval.evaluate(preds_gbt)
mae_gbt  = mae_eval.evaluate(preds_gbt)

best_gbt = gbt_model.bestModel
print("Best params:", {str(k): v for k, v in best_gbt.extractParamMap().items() if 'maxIter' in str(k) or 'maxDepth' in str(k)})

print(f"GBT | RMSE={rmse_gbt:.2f} | R2={r2_gbt:.4f} | MAE={mae_gbt:.2f} | time={time_gbt:.2f}s")

Best params: {'GBTRegressor_3f8670ced36b__maxDepth': 3, 'GBTRegressor_3f8670ced36b__maxIter': 20}
GBT | RMSE=1470.47 | R2=0.2130 | MAE=407.53 | time=198.73s


# Regression summary table

In [ ]:
schema_reg = ["Model", "RMSE", "R2", "MAE", "Train_Time_sec"]

rows_reg = [
    ("Linear Regression", rmse_lr,  r2_lr,  mae_lr,  time_lr),
    ("Decision Tree",     rmse_dt,  r2_dt,  mae_dt,  time_dt),
    ("Random Forest",     rmse_rf,  r2_rf,  mae_rf,  time_rf),
    ("GBT",               rmse_gbt, r2_gbt, mae_gbt, time_gbt),
]


reg_summary = spark.createDataFrame(rows_reg, schema_reg)


print("Sorted by RMSE (lower is better)")
reg_summary.orderBy("RMSE").show(truncate=False)

print("Sorted by R² (higher is better)")
reg_summary.orderBy(reg_summary.R2.desc()).show(truncate=False)

Sorted by RMSE (lower is better)
+-----------------+------------------+-------------------+------------------+------------------+
|Model            |RMSE              |R2                 |MAE               |Train_Time_sec    |
+-----------------+------------------+-------------------+------------------+------------------+
|Random Forest    |1464.8229568445995|0.21907086471886184|418.1170475335348 |128.7243685722351 |
|GBT              |1470.467470088476 |0.21304084232142995|407.52763107458435|198.72674775123596|
|Decision Tree    |1479.4572858325198|0.20338915865006568|425.60503223171736|62.99402618408203 |
|Linear Regression|1487.83225716007  |0.1943446577638248 |456.3234312589696 |109.47206425666809|
+-----------------+------------------+-------------------+------------------+------------------+

Sorted by R² (higher is better)
+-----------------+------------------+-------------------+------------------+------------------+
|Model            |RMSE              |R2                 |MAE

# Saving the 4 models + metadata

In [ ]:
import json

# ----------------------------------
# Save all four best models
# ----------------------------------

#---- Model save locations:
LR_MODEL_PATH = "models/linear_regression"
DT_MODEL_PATH = "models/decision_tree"
RF_MODEL_PATH = "models/random_forest"
GBT_MODEL_PATH = "models/gbt"

# -------------------------------------
# Saving best model from each algorithm
# -------------------------------------

best_lr.write().overwrite().save(LR_MODEL_PATH)
best_dt.write().overwrite().save(DT_MODEL_PATH)
best_rf.write().overwrite().save(RF_MODEL_PATH)
best_gbt.write().overwrite().save(GBT_MODEL_PATH)

print("All 4 models saved ✓")


# --------------------------------------
# Creating coefficient and importance list
# --------------------------------------

lr_coef_list = [(i, float(c)) for i, c in enumerate(best_lr.coefficients)]

rf_fi_list = [(i, float(v)) for i, v in enumerate(best_rf.featureImportances.toArray())]


# ----------------------------------
# Saving Task3 metadata JSON
# ----------------------------------

task3_meta = {
"models": {

    "LinearRegression": {
        "path": LR_MODEL_PATH,
        "rmse": round(rmse_lr, 4),
        "r2": round(r2_lr, 4),
        "mae": round(mae_lr, 4),
        "train_sec": round(time_lr, 1),
        "best_params": str(best_lr.extractParamMap()),
        "coefficients": lr_coef_list
    },

    "DecisionTree": {
        "path": DT_MODEL_PATH,
        "rmse": round(rmse_dt, 4),
        "r2": round(r2_dt, 4),
        "mae": round(mae_dt, 4),
        "train_sec": round(time_dt, 1),
        "best_params": str(best_dt.extractParamMap())
    },

    "RandomForest": {
        "path": RF_MODEL_PATH,
        "rmse": round(rmse_rf, 4),
        "r2": round(r2_rf, 4),
        "mae": round(mae_rf, 4),
        "train_sec": round(time_rf, 1),
        "best_params": str(best_rf.extractParamMap()),
        "feature_importance": rf_fi_list
    },

    "GBT": {
        "path": GBT_MODEL_PATH,
        "rmse": round(rmse_gbt, 4),
        "r2": round(r2_gbt, 4),
        "mae": round(mae_gbt, 4),
        "train_sec": round(time_gbt, 1),
        "best_params": str(best_gbt.extractParamMap())
    }
},

"timestamp": time.strftime("%Y-%m-%dT%H:%M:%S")

}

with open(TASK3_META, "w") as f:
  json.dump(task3_meta, f, indent=2)

print("Task 3 metadata saved ✓")
print("Ready for Task 4 / Task 5")

All 4 models saved ✓
Task 3 metadata saved ✓
Ready for Task 4 / Task 5


## Summary

In this notebook, four regression models were trained and compared using distributed PySpark pipelines. Hyperparameter tuning was performed using CrossValidator, and model performance was evaluated using training time, RMSE, MAE and R².

The trained models are now ready for further evaluation, stability analysis and explainability.